# BNPL Advisor — Model training (EDA → evaluation)

Run `ml_pipeline/scripts/generate_synthetic_data.py` first to create `data/ml_training.csv`.

In [ ]:
import json
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import ConfusionMatrixDisplay, accuracy_score, classification_report, roc_auc_score, roc_curve
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

ROOT = Path('..').resolve()
DATA = ROOT / 'data' / 'ml_training.csv'
MODEL_DIR = ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True)

df = pd.read_csv(DATA)
df.head()

## EDA

In [ ]:
df.describe().T

In [ ]:
sns.countplot(x='label', data=df)
plt.title('Class balance')
plt.show()

In [ ]:
numeric = df.select_dtypes(include=[np.number]).columns.drop('label', errors='ignore')
sns.heatmap(df[numeric].corr(), cmap='coolwarm', center=0)
plt.title('Feature correlation')
plt.show()

## Feature matrix & split

In [ ]:
FEATURE_ORDER = [
    'days_cash_on_hand', 'current_ratio', 'burn_rate_monthly_rm', 'purchase_amount',
    'purchase_to_burn', 'is_digitalisation', 'is_agri', 'monthly_net_cash',
]
X = df[FEATURE_ORDER]
y = df['label']
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

## Hyperparameter tuning (RF example)

In [ ]:
param_grid = {
    'n_estimators': [120, 200],
    'max_depth': [8, 12, None],
    'min_samples_leaf': [2, 4],
}
rf_base = RandomForestClassifier(class_weight='balanced', random_state=42, n_jobs=-1)
gs = GridSearchCV(rf_base, param_grid, cv=3, scoring='roc_auc', n_jobs=-1)
gs.fit(X_train, y_train)
print('Best RF params:', gs.best_params_)
rf_best = gs.best_estimator_

## Train all three models

In [ ]:
lr = LogisticRegression(max_iter=10000, class_weight='balanced', random_state=42)
lr.fit(X_train, y_train)

xgb = XGBClassifier(
    n_estimators=300, max_depth=5, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.85, eval_metric='logloss', random_state=42,
)
xgb.fit(X_train, y_train)

joblib.dump(lr, MODEL_DIR / 'logistic_regression.pkl')
joblib.dump(rf_best, MODEL_DIR / 'random_forest.pkl')
joblib.dump(xgb, MODEL_DIR / 'xgboost.pkl')
(MODEL_DIR / 'feature_order.json').write_text(json.dumps(FEATURE_ORDER), encoding='utf-8')
print('Saved models to', MODEL_DIR)

## Evaluation — confusion matrix & ROC / AUC

In [ ]:
def report(model, name):
    proba = model.predict_proba(X_test)[:, 1]
    pred = (proba >= 0.5).astype(int)
    print(name, 'accuracy', accuracy_score(y_test, pred))
    print(classification_report(y_test, pred))
    print('AUC', roc_auc_score(y_test, proba))
    ConfusionMatrixDisplay.from_predictions(y_test, pred)
    plt.title(f'{name} confusion matrix')
    plt.show()
    fpr, tpr, _ = roc_curve(y_test, proba)
    plt.plot(fpr, tpr, label=name)

plt.figure()
report(lr, 'LogisticRegression')
report(rf_best, 'RandomForest')
report(xgb, 'XGBoost')
plt.plot([0, 1], [0, 1], 'k--')
plt.xlabel('FPR')
plt.ylabel('TPR')
plt.title('ROC curves')
plt.legend()
plt.show()

## SHAP (local explanations)
Use `shap.TreeExplainer(xgb)` on a small sample of `X_test` for waterfall or bar plots.

In [ ]:
import shap

explainer = shap.TreeExplainer(xgb)
sv = explainer.shap_values(X_test.iloc[:50])
shap.summary_plot(sv, X_test.iloc[:50], feature_names=FEATURE_ORDER)